# Simulação de 1 GDL · PIBIC

Oscilador massa–mola–amortecedor com força aplicada à massa.

$$m\ddot{x}+c\dot{x}+kx=F(t), \qquad c=2\zeta\sqrt{mk}$$

Execute as células de cima para baixo. A seção 1 permite alterar a massa, a rigidez, o amortecimento e a excitação. A seção 2 compara as quatro bandas de ruído com a mesma força RMS. A seção 3 reproduz o banco de 1.260 simulações do PIBIC.

As grandezas estão no SI. O impulso ideal é aplicado como salto de velocidade de magnitude J/m em t = 0. A aceleração exportada nesse caso é a parte regular após o salto.

In [ ]:
# @title Importações
import csv
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
# @title Modelo físico e indicadores
DURACAO_S = 64
FS_HZ = 512
AMOSTRAS = DURACAO_S * FS_HZ
AMOSTRAGEM = 20261023
ESCALAS_M = (0.002, 0.005, 0.010)
BANDAS = {
    "subressonante": (0.25, 0.75),
    "ressonante": (0.75, 1.25),
    "suprarressonante": (1.25, 2.50),
    "larga": (0.25, 2.50),
}


def resposta(caso, refinamento=1):
    m, k, c = (float(caso[chave]) for chave in ("m_kg", "k_N_m", "c_N_s_m"))
    tipo = caso["excitacao"]
    tempo = np.arange(AMOSTRAS * refinamento + 1) / (FS_HZ * refinamento)
    alfa = c / (2 * m)
    wd = np.sqrt(k / m - alfa**2)
    x0, v0 = (float(caso[chave]) for chave in ("x0_m", "v0_m_s"))
    if tipo == "degrau":
        forca = np.full_like(tempo, float(caso["F_degrau_N"]))
        xp, vp = forca / k, np.zeros_like(tempo)
    elif tipo == "impulso":
        forca = xp = vp = np.zeros_like(tempo)
        v0 += float(caso["J_impulso_N_s"]) / m
    else:
        rng = np.random.default_rng(int(caso["semente_ruido"]))
        frequencias = np.fft.rfftfreq(AMOSTRAS, 1 / FS_HZ)
        banda = (frequencias >= float(caso["fmin_Hz"])) & (frequencias <= float(caso["fmax_Hz"]))
        if banda.sum() < 2 or banda[-1]:
            raise ValueError("A banda requer pelo menos dois bins e deve ficar abaixo de Nyquist.")
        # FFT de ruído gaussiano branco, com retenção apenas dos bins desejados.
        F = np.fft.rfft(rng.standard_normal(AMOSTRAS))
        F[~banda] = 0
        rms = np.sqrt(2 * np.sum(abs(F)**2)) / AMOSTRAS
        F *= float(caso["F_ruido_rms_N"]) / rms
        jw = 2j * np.pi * frequencias
        X = F / (k + c * jw + m * jw**2)
        n = AMOSTRAS * refinamento
        forca, xp, vp = [
            np.append(amostras, amostras[0])
            for amostras in (np.fft.irfft(espectro * refinamento, n=n) for espectro in (F, X, jw * X))
        ]

    # A resposta homogênea impõe as condições iniciais à solução particular.
    A = x0 - xp[0]
    B = (v0 - vp[0] + alfa * A) / wd
    decaimento = np.exp(-alfa * tempo)
    co, si = np.cos(wd * tempo), np.sin(wd * tempo)
    xh = decaimento * (A * co + B * si)
    x = xp + xh
    v = vp + decaimento * wd * (-A * si + B * co) - alfa * xh
    a = (forca - c * v - k * x) / m
    return tempo, forca, x, v, a


def indicadores(serie):
    tempo, _, *sinais = serie
    rotulos = ("x", "v", "a")
    unidades = ("m", "m_s", "m_s2")
    saida = {}
    for nome, unidade, sinal in zip(rotulos, unidades, sinais):
        quadrado = sinal**2
        integral = (tempo[1] - tempo[0]) * (quadrado.sum() - (quadrado[0] + quadrado[-1]) / 2)
        saida[f"{nome}_rms_{unidade}"] = float(np.sqrt(integral / tempo[-1]))
        saida[f"{nome}_pico_abs_{unidade}"] = float(np.max(abs(sinal)))
    return saida

## 1. Simular um caso

Escolha degrau, impulso ou ruído. Para ruído, as bandas são relativas à frequência natural do sistema. O filtro é aplicado aos coeficientes da FFT de um ruído branco gaussiano e a força é ajustada ao RMS escolhido. A resposta inclui o transiente das condições iniciais.

In [ ]:
# @title Parâmetros da simulação
excitacao = "ruido" # @param ["degrau", "impulso", "ruido"]
banda = "larga" # @param ["subressonante", "ressonante", "suprarressonante", "larga"]
massa_kg = 40.0 # @param {type:"number"}
rigidez_N_m = 40000.0 # @param {type:"number"}
zeta = 0.02 # @param {type:"number"}
forca_degrau_N = 80.0 # @param {type:"number"}
impulso_N_s = 2.529822128 # @param {type:"number"}
forca_ruido_rms_N = 80.0 # @param {type:"number"}
semente_ruido = 20261034 # @param {type:"integer"}
janela_grafico_s = 8.0 # @param {type:"number"}

if not (np.isfinite([massa_kg, rigidez_N_m, zeta]).all()
        and massa_kg > 0 and rigidez_N_m > 0 and 0 < zeta < 1):
    raise ValueError("Use massa e rigidez positivas e amortecimento relativo estritamente entre 0 e 1.")
if excitacao not in ("degrau", "impulso", "ruido"):
    raise ValueError("Escolha degrau, impulso ou ruido.")
if not np.isfinite([forca_degrau_N, impulso_N_s, forca_ruido_rms_N]).all():
    raise ValueError("Use valores finitos para as forças e o impulso.")
if forca_ruido_rms_N <= 0 or semente_ruido < 0:
    raise ValueError("Use força RMS positiva e semente não negativa.")
if not 0 < janela_grafico_s <= DURACAO_S:
    raise ValueError(f"Escolha uma janela maior que zero e de até {DURACAO_S} segundos.")

omega_n = np.sqrt(rigidez_N_m / massa_kg)
fn = omega_n / (2 * np.pi)
caso = {
    "m_kg": massa_kg, "k_N_m": rigidez_N_m,
    "c_N_s_m": 2 * zeta * massa_kg * omega_n,
    "x0_m": 0.0, "v0_m_s": 0.0, "excitacao": excitacao,
    "F_degrau_N": forca_degrau_N, "J_impulso_N_s": impulso_N_s,
    "F_ruido_rms_N": forca_ruido_rms_N, "semente_ruido": semente_ruido,
    "fmin_Hz": fn * BANDAS[banda][0], "fmax_Hz": fn * BANDAS[banda][1],
}
if excitacao == "ruido" and caso["fmax_Hz"] >= FS_HZ / 2:
    raise ValueError("A banda deve ficar abaixo da frequência de Nyquist (256 Hz).")

serie = resposta(caso)
t, F, x, v, a = serie
print(f"Frequência natural: {fn:.3f} Hz | Duração: {DURACAO_S} s | Amostragem: {FS_HZ} Hz")
if excitacao == "ruido":
    print(f"Banda: {caso['fmin_Hz']:.3f} a {caso['fmax_Hz']:.3f} Hz")
display(pd.DataFrame([indicadores(serie)]))

In [ ]:
# @title Gráficos e exportação da série temporal
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(10, 8), layout="constrained")
for eixo, valores, rotulo in zip(axes, [F, x * 1000, v, a],
                               ["Força regular (N)", "Deslocamento (mm)",
                                "Velocidade (m/s)", "Aceleração regular (m/s²)"]):
    eixo.plot(t, valores, linewidth=1.0)
    eixo.set_ylabel(rotulo)
    eixo.grid(alpha=0.25)
if excitacao == "impulso":
    axes[0].set_title(f"Impulso em t = 0: J = {impulso_N_s:g} N·s")
axes[-1].set(xlabel="Tempo (s)", xlim=(0, janela_grafico_s))
plt.show()

arquivo_serie = Path(f"serie_1gdl_{excitacao}.csv")
pd.DataFrame(dict(t_s=t, F_regular_N=F, x_m=x, v_m_s=v, a_regular_m_s2=a)).to_csv(
    arquivo_serie, index=False
)
print(f"Série salva em {arquivo_serie}.")

## 2. Comparar as bandas

A comparação usa o mesmo sistema, a mesma semente e a mesma força RMS em todas as faixas. O gráfico mostra o espectro da **entrada** e o deslocamento RMS da **resposta** calculado nos 64 segundos.

In [ ]:
# @title Espectro da força e deslocamento RMS por banda
fig, (ax_psd, ax_rms) = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
comparacao = []
for nome, (r_min, r_max) in BANDAS.items():
    f_min, f_max = fn * r_min, fn * r_max
    if f_max >= FS_HZ / 2:
        raise ValueError("Com esse sistema, pelo menos uma banda atinge ou passa de 256 Hz.")
    teste = {**caso, "excitacao": "ruido", "fmin_Hz": f_min, "fmax_Hz": f_max}
    serie_banda = resposta(teste)
    amostras_forca = serie_banda[1][:-1]
    n = len(amostras_forca)
    f = np.fft.rfftfreq(n, d=1 / FS_HZ)
    psd = np.abs(np.fft.rfft(amostras_forca))**2 / (FS_HZ * n)
    psd[1:-1] *= 2
    dentro = (f >= f_min) & (f <= f_max)
    ax_psd.plot(f[dentro] / fn, psd[dentro], label=nome, alpha=0.75)
    comparacao.append({
        "banda": nome, "fmin_Hz": f_min, "fmax_Hz": f_max,
        "F_rms_N": np.sqrt(np.mean(amostras_forca**2)),
        "x_rms_mm": 1000 * indicadores(serie_banda)["x_rms_m"],
    })
comparacao = pd.DataFrame(comparacao)
ax_psd.axvline(1, linestyle="--", color="0.4", linewidth=1)
ax_psd.set(xlabel="Frequência normalizada f/fₙ", ylabel="PSD da força (N²/Hz)")
ax_psd.legend(fontsize=8)
ax_rms.bar(comparacao["banda"], comparacao["x_rms_mm"], color="#226b85")
ax_rms.set_ylabel("Deslocamento RMS (mm)")
ax_rms.tick_params(axis="x", rotation=25)
plt.show()
display(comparacao.round(4))

## 3. Reproduzir o banco do PIBIC

Esta etapa é opcional. Usa o CSV harmônico do repositório como fonte dos sistemas de referência. Gera **90 casos de degrau, 90 de impulso e 270 em cada uma das quatro bandas de ruído**. Os parâmetros da seção 1 são para a simulação individual. Este lote usa o desenho original do banco.

In [ ]:
# @title Leitura dos sistemas de referência e geração das combinações
def sistemas_do_banco(caminho):
    sistemas = {}
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        for linha in csv.DictReader(arquivo):
            if linha["split"] == "teste_cego_rao":
                parametros = tuple(float(linha[chave]) for chave in ("m", "k", "zeta"))
                chave = tuple(round(p, 10) for p in parametros)
                sistema = sistemas.setdefault(chave, [parametros, []])
                sistema[1].append(linha["id"])
    if not sistemas:
        raise ValueError("O CSV deve conter o split teste_cego_rao e as colunas id, m, k e zeta.")
    return list(sistemas.values())


def casos(sistemas):
    rng = np.random.default_rng(AMOSTRAGEM)
    for grupo, ((m0, k0, zeta), origens) in enumerate(sistemas, 1):
        fatores = [(1.0, 1.0), *rng.uniform(0.85, 1.15, size=(2, 2))]
        for variante, (fm, fk) in enumerate(fatores):
            m, k = m0 * fm, k0 * fk
            wn = np.sqrt(k / m)
            fn = wn / (2 * np.pi)
            for escala_id, escala in enumerate(ESCALAS_M, 1):
                base = {
                    "grupo_origem": f"G{grupo:02d}",
                    "sistema_id": f"G{grupo:02d}-V{variante}",
                    "casos_rao_origem": "|".join(origens),
                    "m_kg": m, "k_N_m": k, "c_N_s_m": 2 * zeta * m * wn,
                    "zeta": zeta, "fn_Hz": fn, "escala_ref_m": escala,
                    "tf_s": DURACAO_S, "fs_Hz": FS_HZ, "x0_m": 0.0, "v0_m_s": 0.0,
                }
                prefixo = f"{base['sistema_id']}-A{escala_id}"
                for tipo, banda, repeticao in [
                    ("degrau", "", 0), ("impulso", "", 0),
                    *(( "ruido", faixa, rep) for faixa in BANDAS for rep in range(1, 4)),
                ]:
                    limites = BANDAS.get(banda)
                    semente = AMOSTRAGEM + grupo * 10 + repeticao if limites else None
                    yield {
                        "id_simulacao": "-".join(filter(None, (prefixo, tipo, banda, str(repeticao) if repeticao else ""))),
                        **base, "excitacao": tipo, "banda": banda,
                        "semente_ruido": semente,
                        "fmin_Hz": fn * limites[0] if limites else None,
                        "fmax_Hz": fn * limites[1] if limites else None,
                        "F_degrau_N": k * escala if tipo == "degrau" else None,
                        "J_impulso_N_s": m * wn * escala if tipo == "impulso" else None,
                        "F_ruido_rms_N": k * escala if tipo == "ruido" else None,
                    }

In [ ]:
# @title Gerar o CSV completo
origem = Path("dataset_completo_treinamento_suspensao_1dof.csv")
origem_url = (
    "https://raw.githubusercontent.com/tedescohf19/pibic-suspensao-veicular/"
    "09a22cf3eb329079d2b7cc44bbb77c7830e6748c/dados/1gdl/harmonico/"
    "dataset_completo_treinamento_suspensao_1dof.csv"
)
if not origem.exists():
    urlretrieve(origem_url, origem)

linhas = [{**entrada, **indicadores(resposta(entrada))}
          for entrada in casos(sistemas_do_banco(origem))]
arquivo_banco = Path("banco_1gdl_excitacoes.csv")
with arquivo_banco.open("w", encoding="utf-8", newline="") as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=linhas[0].keys())
    writer.writeheader()
    writer.writerows(linhas)
banco = pd.DataFrame(linhas)
print(f"{len(linhas)} simulações salvas em {arquivo_banco}.")
display(banco.groupby(["excitacao", "banda"]).size().rename("simulações"))

## 4. Análise determinística e comparação com RK4

O caso de referência usa m = 40 kg, k = 40.000 N/m e ζ = 2%. O degrau de 80 N tem deslocamento estático de 2 mm. O impulso ideal de aproximadamente 2,53 N·s impõe um salto inicial de aproximadamente 0,06325 m/s. O gráfico mostra o deslocamento nas duas respostas.

In [ ]:
# @title Respostas ao degrau e ao impulso
m_ref, k_ref, zeta_ref = 40.0, 40000.0, 0.02
wn_ref = np.sqrt(k_ref / m_ref)
escala_ref = 0.002
caso_ref = dict(m_kg=m_ref, k_N_m=k_ref, c_N_s_m=2*zeta_ref*m_ref*wn_ref,
                x0_m=0.0, v0_m_s=0.0, F_degrau_N=k_ref*escala_ref,
                J_impulso_N_s=m_ref*wn_ref*escala_ref)
fig, eixos = plt.subplots(2, 1, sharex=True, figsize=(8.6, 4.6), layout="constrained")
resumo_deterministico = []
for eixo, tipo, cor in zip(eixos, ("degrau", "impulso"), ("#126777", "#9A4D24")):
    s = resposta({**caso_ref, "excitacao": tipo})
    tempo, _, deslocamento, velocidade, _ = s
    eixo.plot(tempo, 1000 * deslocamento, color=cor, label=tipo.capitalize())
    equilibrio = escala_ref if tipo == "degrau" else 0.0
    eixo.axhline(1000 * equilibrio, linestyle="--", color="0.5", label="Equilíbrio")
    eixo.set(ylabel="Deslocamento (mm)", xlim=(0, 4))
    eixo.grid(alpha=0.2)
    eixo.legend(loc="upper right")
    resumo_deterministico.append({
        "excitacao": tipo, "equilibrio_mm": 1000*equilibrio,
        "velocidade_inicial_m_s": velocidade[0], **indicadores(s),
    })
eixos[-1].set_xlabel("Tempo (s)")
fig.savefig("respostas_deterministicas.png", dpi=170)
plt.show()
deterministico = pd.DataFrame(resumo_deterministico)
display(deterministico.round(6))

In [ ]:
# @title Integrador RK4 para verificação independente
def rk4(caso, f):
    m, c, k = (float(caso[key]) for key in ("m_kg", "c_N_s_m", "k_N_m"))
    x, v = float(caso["x0_m"]), float(caso["v0_m_s"])
    if caso["excitacao"] == "impulso":
        v += float(caso["J_impulso_N_s"]) / m
    sol = np.empty((AMOSTRAS + 1, 2))
    sol[0] = x, v
    h = 1 / FS_HZ
    for i in range(AMOSTRAS):
        f0, f1, f2 = f[2*i:2*i+3]
        a1 = (f0-c*v-k*x)/m
        v2 = v+h*a1/2
        a2 = (f1-c*v2-k*(x+h*v/2))/m
        v3 = v+h*a2/2
        a3 = (f1-c*v3-k*(x+h*v2/2))/m
        v4 = v+h*a3
        a4 = (f2-c*v4-k*(x+h*v3))/m
        x += h*(v+2*v2+2*v3+v4)/6
        v += h*(a1+2*a2+2*a3+a4)/6
        sol[i+1] = x, v
    return sol

In [ ]:
# @title Verificar 12 casos e medir a sensibilidade ao passo
rotulo = banco["banda"].where(banco["excitacao"].eq("ruido"), banco["excitacao"])
registros = []
for categoria, bloco in banco.assign(categoria=rotulo).groupby("categoria", sort=False):
    ordenado = bloco.sort_values("fn_Hz", kind="stable")
    for extremo, linha in (("mínima", ordenado.iloc[0]), ("máxima", ordenado.iloc[-1])):
        caso_teste = linha.to_dict()
        refinada = resposta(caso_teste, refinamento=2)
        _, forca, x, v, _ = refinada
        referencia = np.column_stack((x[::2], v[::2]))
        erro = np.max(np.abs(rk4(caso_teste, forca) - referencia), axis=0)
        erro *= 100 / np.max(np.abs(referencia), axis=0)
        indicadores_refinados = indicadores(refinada)
        variacoes = {
            chave: 100 * abs(valor - caso_teste[chave]) / abs(valor)
            for chave, valor in indicadores_refinados.items()
        }
        registros.append({
            "categoria": categoria, "fn_extrema": extremo, "id_simulacao": caso_teste["id_simulacao"],
            "fn_Hz": caso_teste["fn_Hz"], "erro_rk4_x_pct": erro[0], "erro_rk4_v_pct": erro[1],
            "variacao_pico_max_pct": max(v for k, v in variacoes.items() if "pico" in k),
            "variacao_rms_max_pct": max(v for k, v in variacoes.items() if "rms" in k),
        })
validacao = pd.DataFrame(registros)
assert len(validacao) == 12 and validacao["erro_rk4_x_pct"].max() < 1
display(validacao.round(5))
validacao.to_csv("validacao_rk4_1gdl.csv", index=False)

In [ ]:
# @title Resumir as faixas e exportar resultados
amostras_ruido = banco.loc[banco["excitacao"] == "ruido"].copy()
amostras_ruido["x_rms_normalizado"] = amostras_ruido["x_rms_m"] / amostras_ruido["escala_ref_m"]
ordem = list(BANDAS)
grupos = amostras_ruido.groupby("banda")["x_rms_normalizado"]
resumo_faixas = grupos.agg(casos="size", mediana="median").reindex(ordem)
resumo_faixas["p25"] = grupos.quantile(.25)
resumo_faixas["p75"] = grupos.quantile(.75)
pares = amostras_ruido.pivot(
    index=["sistema_id", "escala_ref_m", "semente_ruido"],
    columns="banda", values="x_rms_normalizado",
)
assert len(pares) == 270 and not pares.isna().any().any()
resumo_faixas["vezes_maior_rms"] = pares.idxmax(axis=1).value_counts().reindex(ordem, fill_value=0)
display(resumo_faixas.round(3))
resumo_faixas.to_csv("resumo_faixas_1gdl.csv")
fig, ax = plt.subplots(figsize=(8.6, 3.8), layout="constrained")
rotulos = ["Subressonante", "Ressonante", "Suprarressonante", "Banda larga"]
boxes = ax.boxplot([grupos.get_group(nome) for nome in ordem], patch_artist=True, showfliers=False)
for caixa, cor in zip(boxes["boxes"], ("#9FD3D8", "#F2C68C", "#BADBC7", "#A5C6EA")):
    caixa.set_facecolor(cor)
ax.set_xticks(range(1, 5), rotulos)
ax.set_ylabel("Deslocamento RMS / escala de referência")
ax.grid(axis="y", alpha=.25)
fig.savefig("comparacao_bandas.png", dpi=170)
plt.show()
print("Estudo concluído: 1.260 simulações, 4 bandas e 12 verificações RK4.")